# Projeto - Simulando a Opinião Pública
## Tema: Democracia
## Modelo escolhido: Gemma 3, 1B de parâmetros

## Primeiros passos
### Antes de executar as células de código, siga os passos a seguir
**Baixar e preparar Ollama:**
1. Escolha o comando apropriado para o seu SO (MacOS, Linux, Windows) no site do [Ollama](https://ollama.com/download)
> Para Linux/MacOS, pode ser necessário executar o comando abaixo
> ```bash
>  sudo apt-get install zstd
> ```
2.  Execute o servidor do Ollama localmente, o que deve abrir a porta `localhost:11434`
    ```bash
    ollama serve
    ```
> OBS: Se retornar que a porta já está em uso, é porque após o download, já foi executado automaticamente o comando anterior. Para confirmar, acesse `http://localhost:11434` no navegador e veja se retorna *Ollama is running* 
3. Execute o comando abaixo para baixar e executar o modelo:
    ```bash
    ollama run gemma3:1b "preload" > /dev/null
    ```
> OBS: Se não funcionar, execute `ollama pull gemma3:1b antes`

**Preparar ambiente virtual do Python:**<br><br>
Para baixar as dependências apenas para esse projeto (e não de forma fixa na sua máquina) execute essas instruções:
```bash
python3 -m venv .venv;
source .venv/bin/activate;
```

### Baixando bibliotecas necessárias

* **Pandas:** Para trabalhar com a base de dados
* **PyReadStat:** Para trabalhar com a base de dados, permitindo ler arquivo .sav e complementar o pandas
* **Ollama:** API do LLM
* **Pydantic + Typing**: Validação de formato de saída JSON

In [16]:
# Instalar a lib de Python para Ollama
!pip install ollama

In [17]:
# Instalar pndas
!pip install pandas

In [18]:
# Instalar a lib para ler arquivos .sav
!pip install pyreadstat

In [19]:
!pip install pydantic

### Desenvolvimento da conversa com o modelo

#### Parte 1) Selecionar variáveis mais relevantes
O primeiro passo é pedir ao LLM reconhecer quais são as variáveis mais relevantes para ele na hora de simular o participante.

Para isso, para cada pergunta, será enviado um prompt indicando essa tarefa, as variáveis com valores possíveis, a pergunta e um JSON Schema de resposta esperada.

Gradualmente, as características devem ser incluídas, não todas de uma vez. (TODO)

##### Entrada
**Prompt de Sistema**
> "*Você é um assistente de raciocínio treinado para entender como características pessoais moldam crenças. Sua tarefa é analisar características e selecionar as mais preditivas para o tópico de Democracia.*
>
> *Você receberá uma pergunta, suas possíveis respostas e se é de múltipla escolha ou escolha única, e deverá retornar uma lista de características que julgar que são as mais relevantes para responder a pergunta.*
> *Características possíveis: <lista de características, ignorando as outras perguntas>*

**Prompt de Entrada**
> *Pergunta: ...,Respostas Possíveis: [...], Múltipla Escolhda: Sim | Não*


##### Saída
```json
  {
    "question" : {
      "type": "string"
    },
    "caracteristicas_selecionadas": {
        "type": "array",
        "items": {
          "type": "string"
        },
        "minItems": 1,
        "uniqueItems": true,
        "default": [
          "Nenhuma é relevante"
        ]
      },
  }
```


In [32]:
import pandas as pd

df = pd.read_spss("04832.SAV")

#Investigando as características dos participantes e seus valores únicos
print("Colunas possíveis:", list(df.columns))

features_ignore = ["ID_Ipec", "DATA_ENTREVISTA", "TIPO_COLETA","IDADE","REND1","REND2"]

#Excluindo as perguntas e características irrelevantes
participant_features = list(df.columns[~df.columns.str.contains(r"P\d\D*")])
feature_renda = pd.concat([df['REND1'].astype(str), df['REND2'].astype(str)], ignore_index=True).rename('RENDA')
participant_features = [feature for feature in participant_features if feature not in features_ignore]
participant_features_values = {feature: list(df[feature].unique()) for feature in participant_features}
participant_features.append(str(feature_renda.name))
participant_features_values['RENDA'] = list(feature_renda.unique())
print("Características dos participantes:", participant_features)
print("Valores únicos para cada característica:")
for feature, values in participant_features_values.items():
    print(f"  {feature}: {values}")

Colunas possíveis: ['SEXO', 'IDADE', 'FX_ID', 'ESCOLARIDADE', 'P1A', 'P1B', 'P1C', 'P2_1', 'P2_2', 'P2_3', 'P3_1', 'P3_2', 'P3_3', 'P3_4', 'P3_5', 'P3_6', 'P4', 'RACA', 'RELIGIAO', 'REND1', 'REND2', 'REGIAO', 'COND', 'PORTE', 'ID_Ipec', 'DATA_ENTREVISTA', 'TIPO_COLETA']
Características dos participantes: ['SEXO', 'FX_ID', 'ESCOLARIDADE', 'RACA', 'RELIGIAO', 'REGIAO', 'COND', 'PORTE', 'RENDA']
Valores únicos para cada característica:
  SEXO: ['FEM', 'MAS']
  FX_ID: ['25 A 34', '65 E MAIS', '45 A 54', '55 A 64', '35 A 44', '18 A 24', '16 E 17']
  ESCOLARIDADE: ['Superior completo', '1ª série (ou 2º ano)', '3ª série', '3ª série (ou 4º ano)', '4ª série (ou 5º ano)', '5ª série (ou 6º ano)', 'Superior incompleto', '2ª série (ou 3º ano)', '2ª série', 'Sabe ler/ escrever, mas não cursou escola', '8ª série (ou 9º ano)', 'Analfabeto', '1ª série', '7ª série (ou 8º ano)', '6ª série (ou 7º ano)', 'Pré-escola (ou 1º ano)']
  RACA: ['Branca', 'Parda', 'Preta', 'Indígena', 'Amarela']
  RELIGIAO: ['Out

In [38]:
questions = [
        {
            "codigo": "P11",
            "pergunta": "Você se lembra em quem votou para Deputado(a) Estadual nas eleições gerais de 2022?",
            "possiveis_respostas": ["Sim", "Não", "Não Votou"],
            "multipla_escolha": False
        },
        {
            "codigo": "P12",
            "pergunta": "Você se lembra em quem votou para Deputado(a) Federal nas eleições gerais de 2022?",
            "possiveis_respostas": ["Sim", "Não", "Não Votou"],
            "multipla_escolha": False
        },
        {
            "codigo": "P13",
            "pergunta": "Você se lembra em quem votou para Senador(a) Federal nas eleições gerais de 2022?",
            "possiveis_respostas": ["Sim", "Não", "Não Votou"],
            "multipla_escolha": False
        },
        {
            "codigo": "P3",
            "pergunta": " Algumas pessoas dizem que a divulgação de fake news, ou seja, de notícias ou conteúdos falsos podem prejudicar a democracia. "
                        "Pensando nisso, quais dessas opções você acredita que poderiam contribuir no combate à divulgação de fake news? ",
            "possiveis_respostas": [
                "Ampliar a regulamentação, as regras a serem cumpridas pelas plataformas digitais (empresas de tecnologia como Facebook, Youtube, WhatsApp, Twitter/X, etc.)",
                "Responsabilizar e punir empresas de tecnologia e de comunicação que não removerem postagens com notícias ou conteúdos falsos",
                "Ampliar a regulamentação, as regras a serem cumpridas pelos usuários que divulgam ou compartilham fake news, criadas por eles próprios ou por terceiros",
                "Responsabilizar e punir os usuários que divulgam ou compartilham postagens com notícias ou conteúdos falsos",
                "Ampliar a regulamentação, as regras a serem cumpridas por políticos/candidatos que divulgam ou compartilham fake news, criadas por eles próprios ou por terceiros",
                "Responsabilizar, punir ou cassar políticos/candidatos que divulgam ou compartilham postagens com notícias ou conteúdos falsos",
                "Não sabe",
                "Não respondeu"
            ],
            "multipla_escolha": True
        }
]

In [34]:
from typing import List
from pydantic import BaseModel, Field

class ExpectedAnswerVariables(BaseModel):
    caracteristicas_selecionadas: List[str] = Field(
        min_length=1,
        description="Lista de características selecionadas pelo modelo, que devem ser as mais relevantes para responder a pergunta. As características devem ser escolhidas entre as seguintes opções: SEXO, FX_ID, ESCOLARIDADE, RAÇA, RENDA, REGIÃO, RELIGIÃO."
    )

In [ ]:
import ollama
import json


#Definindo Host do Ollama
ollama_host = 'http://localhost:11434'
ollama.Client(host=ollama_host)

participant_features_prompt = """
SEXO: Masculino, Feminino
FX_ID: Faixa de idade do participante, em décadas 
ESCOLARIDADE: Grau de escolaridade do participante
RAÇA: Raça do participante
RENDA: Faixa de renda mensal do participante, em milhares de reais
REGIÃO: Região geográfica do país onde o participante reside
RELIGIÃO: Religião do participante
COND: Se o participante vive na Capital, Interior ou Periferia
PORTE: Patrimônio financeiro do participante, em milhares de reais
"""

# Usando prompt de sistema
system_prompt = f"""
Você é um assistente de raciocínio treinado para entender como características pessoais moldam crenças. Sua tarefa é analisar características e selecionar as mais preditivas para o tópico de Democracia.*
Você receberá uma pergunta, suas possíveis respostas e se é de múltipla escolha ou escolha única, e deverá retornar uma lista de características que julgar que são as mais relevantes para responder a pergunta.*
Características possíveis: \n{participant_features_prompt}
IMPORTANTE: 
- RESPONDA APENAS COM O NOME DA CARACTERÍSTICA SELECIONADA, o qual está em LETRAS CAPITAIS na lista acima.
- Você deve APENAS RETORNAR A LISTA DE CARACTERÍSTICAS SELECIONADAS, sem explicações ou justificativas. 
- NÃO adicione nenhuma caracterísitca que não esteja na lista.
"""

features_selected_for_question = {}

# Chama da API do Ollama
for question in questions:
    user_question = f"""
    Pergunta: {question['pergunta']}
    Possíveis respostas: {', '.join(question['possiveis_respostas'])}
    Múltipla escolha: {'Sim' if question['multipla_escolha'] else 'Não'}
    """
    try:
        response = ollama.chat(
            model='gemma3:1b',
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_question},
            ],
            format=ExpectedAnswerVariables.model_json_schema(),
            options={"temperature": 0.3}
        )
        print("#" * 50)
        print(f"Pergunta Lida: {question['pergunta']}")
        print("Resposta do Ollama:\n")
        features_selected_for_question[question['codigo']] = json.loads(response['message']['content'])['caracteristicas_selecionadas']
        print(f"Características selecionadas: {features_selected_for_question[question['codigo']]}")
        print("#" * 50)
    except Exception as e:
        print(f"Ocorreu um erro ao conectar com o Ollama: {e}")


##################################################
Pergunta Lida: Você se lembra em quem votou para Deputado(a) Estadual nas eleições gerais de 2022?
Resposta do Ollama:

['SEXO', 'REGIÃO']
##################################################
##################################################
Pergunta Lida: Você se lembra em quem votou para Deputado(a) Federal nas eleições gerais de 2022?
Resposta do Ollama:

['SEXO', 'RAENA', 'REGIÃO']
##################################################
##################################################
Pergunta Lida: Você se lembra em quem votou para Senador(a) Federal nas eleições gerais de 2022?
Resposta do Ollama:

['SEXO', 'RENDA', 'REGIÃO']
##################################################
##################################################
Pergunta Lida:  Algumas pessoas dizem que a divulgação de fake news, ou seja, de notícias ou conteúdos falsos podem prejudicar a democracia. Pensando nisso, quais dessas opções você acredita que poderiam contrib

#### Parte 2) Simular participante
O segundo passo é pedir ao LLM responder às perguntas, e depois comparar com as características que ele escolheu para ele na hora de simular o participante.

Para isso, para cada pergunta, será enviado um prompt indicando essa tarefa, o perfil do respondente e a pergunta

Gradualmente, as características devem ser incluídas, não todas de uma vez.

##### Entrada
**Prompt de Sistema**
> "*Você é um participante de um questionário sobre Democracia. Sua tarefa é responder a pergunta fornecida, considerando o perfil passdo do participante que você está
> representando.*
> *IMPORTANTE: Responda apenas com a alternativa escolhida (ou com as alternativas, se for de múltipla escolha). NUNCA responda com uma alternativa que não esteja na lista

**Prompt de Entrada**
> *Perfil: [...]*
>  *Pergunta: ...,Respostas Possíveis: [...], Múltipla Escolhda: Sim | Não*

##### Saída
**Escolha única**
```json
  {
    "pergunta" : {
      "type": "string"
    },
    "resposta": {
        "type": "string"
    }
  }
```

**Múltipla escolha**
```json
  {
    "pergunta" : {
      "type": "string"
    },
    "respostas": {
        "type": "array",
        "items": {
          "type": "string"
        },
        "minItems": 1,
        "uniqueItems": true,
      },
  }
```


In [24]:
system_prompt = f"""
Você é um participante de um questionário sobre Democracia. Sua tarefa é responder à pergunta fornecida, considerando o perfil do participante que você está representando.
IMPORTANTE: Responda apenas com a alternativa escolhida (ou com as alternativas, se for de múltipla escolha). NUNCA responda com uma alternativa que não esteja na lista
"""

for question in questions:
    user_question = f"""
    Pergunta: {question['pergunta']}
    Possíveis respostas: {', '.join(question['possiveis_respostas'])}
    Múltipla escolha: {'Sim' if question['multipla_escolha'] else 'Não'}
    """
    try:
        response = ollama.chat(
            model='gemma3:1b',
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_question},
            ],
            format=ExpectedAnswerVariables.model_json_schema(),
            options={"temperature": 0.3}
        )
        print("#" * 50)
        print(f"Pergunta Lida: {question['pergunta']}")
        print("Resposta do Ollama:\n")
        print(response['message']['content'])
        print("#" * 50)
    except Exception as e:
        print(f"Ocorreu um erro ao conectar com o Ollama: {e}")
        print("Por favor, certifique-se de que o servidor Ollama está em execução e o modelo 'gemm-3-1B' está baixado.")
        print(f"Você pode tentar iniciar o servidor Ollama e baixar o modelo com: 'ollama run gemm-3-1B' em seu terminal.")


##################################################
Pergunta Lida: Você se lembra em quem votou para Deputado(a) Estadual nas eleições gerais de 2022?
Resposta do Ollama:

{
  "variaveis_selecionadas": [
    "Não",
    "Não Votou"
  ]
}
##################################################
##################################################
Pergunta Lida: Você se lembra em quem votou para Deputado(a) Federal nas eleições gerais de 2022?
Resposta do Ollama:

{
  "variaveis_selecionadas": [
    "Não",
    "Não",
    "Não"
  ]
}

##################################################
##################################################
Pergunta Lida: Você se lembra em quem votou para Senador(a) Federal nas eleições gerais de 2022?
Resposta do Ollama:

{
  "variaveis_selecionadas": ["Não"]
}
##################################################
##################################################
Pergunta Lida:  Algumas pessoas dizem que a divulgação de fake news, ou seja, de notícias ou conteúdos falsos 

## Parte 3) Avaliar Desempenho

### F1-Score/Acurácia
### Distribuição das respostas
### Explicabilidade (Importância das variáveis)

In [25]:
# TODO